<a href="https://colab.research.google.com/github/okayren/Palepal-Skripsi/blob/master/model_training/Palepal_dataset_konjungtiva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sun Jul 12 14:32:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install ultralytics imagehash -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 30.1 MB/s eta 0:00:00


In [ ]:
import os
import shutil
import random
import yaml
import re
import zipfile

ZIP_PATH = "/content/konjungtiva 3.v1-konjungtiva-mata.yolov8.zip"
RAW_ROOT = "/content/dataset_konjungtiva_od"           # folder hasil ekstrak zip
OUTPUT_ROOT = "/content/dataset_konjungtiva_od_cls"    # dataset classification hasil konversi

In [ ]:
for p in [RAW_ROOT, OUTPUT_ROOT]:
    if os.path.exists(p):
        print(f"Menghapus folder lama {p}...")
        shutil.rmtree(p)

print("Mengekstrak file ZIP...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(RAW_ROOT)
print("Ekstrak selesai!")

!find {RAW_ROOT} -maxdepth 2 -type d

Mengekstrak file ZIP...
Ekstrak selesai!
/content/dataset_konjungtiva_od
/content/dataset_konjungtiva_od/train
/content/dataset_konjungtiva_od/train/labels
/content/dataset_konjungtiva_od/train/images
/content/dataset_konjungtiva_od/valid
/content/dataset_konjungtiva_od/valid/labels
/content/dataset_konjungtiva_od/valid/images
/content/dataset_konjungtiva_od/test
/content/dataset_konjungtiva_od/test/labels
/content/dataset_konjungtiva_od/test/images


In [ ]:
with open(os.path.join(RAW_ROOT, "data.yaml"), "r") as f:
    data_yaml = yaml.safe_load(f)
class_names = data_yaml["names"]
print("Urutan kelas dari data.yaml:", class_names)

EXCLUDED_CLASSES = ["\\"]  # kelas noise yang dibuang

split_map = {"train": "train", "valid": "val", "test": "test"}

stats = {"total_images": 0, "skipped_no_label": 0, "skipped_mixed_or_excluded": 0, "copied": 0}
per_class_count = {}

for raw_split, out_split in split_map.items():
    img_dir = os.path.join(RAW_ROOT, raw_split, "images")
    label_dir = os.path.join(RAW_ROOT, raw_split, "labels")
    if not os.path.exists(img_dir):
        continue

    for fname in os.listdir(img_dir):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        stats["total_images"] += 1

        label_fname = os.path.splitext(fname)[0] + ".txt"
        label_path = os.path.join(label_dir, label_fname)

        if not os.path.exists(label_path):
            stats["skipped_no_label"] += 1
            continue

        with open(label_path, "r") as f:
            lines = [l.strip() for l in f if l.strip()]

        if not lines:
            stats["skipped_no_label"] += 1
            continue

        class_ids = set()
        for line in lines:
            class_id = int(line.split()[0])
            class_ids.add(class_id)

        class_labels = {class_names[cid] for cid in class_ids}

        if class_labels & set(EXCLUDED_CLASSES):
            stats["skipped_mixed_or_excluded"] += 1
            continue

        if len(class_labels) > 1:
            stats["skipped_mixed_or_excluded"] += 1
            continue

        final_label = list(class_labels)[0]

        out_dir = os.path.join(OUTPUT_ROOT, out_split, final_label)
        os.makedirs(out_dir, exist_ok=True)
        shutil.copy(os.path.join(img_dir, fname), os.path.join(out_dir, fname))

        stats["copied"] += 1
        per_class_count[final_label] = per_class_count.get(final_label, 0) + 1

print("\n=== HASIL KONVERSI OD -> CLASSIFICATION ===")
print(f"Total gambar ditemukan    : {stats['total_images']}")
print(f"Dilewati (tanpa label)    : {stats['skipped_no_label']}")
print(f"Dilewati (excluded/ambigu): {stats['skipped_mixed_or_excluded']}")
print(f"Berhasil dikonversi       : {stats['copied']}")

print("\n=== DISTRIBUSI KELAS HASIL KONVERSI (semua split digabung) ===")
for cls, n in per_class_count.items():
    print(f"  {cls}: {n} gambar")

Urutan kelas dari data.yaml: ['Anemia', 'Normal']

=== HASIL KONVERSI OD -> CLASSIFICATION ===
Total gambar ditemukan    : 1983
Dilewati (tanpa label)    : 10
Dilewati (excluded/ambigu): 0
Berhasil dikonversi       : 1973

=== DISTRIBUSI KELAS HASIL KONVERSI (semua split digabung) ===
  Normal: 984 gambar
  Anemia: 989 gambar


In [ ]:
import imagehash
from PIL import Image

HASH_THRESHOLD = 5

def compute_hashes(folder):
    hashes = {}
    if not os.path.exists(folder):
        return hashes
    for f in os.listdir(folder):
        if not f.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        try:
            hashes[f] = imagehash.phash(Image.open(os.path.join(folder, f)))
        except Exception as e:
            print(f"Gagal baca {f}: {e}")
    return hashes

print(f"\n{'='*60}")
print("CEK LEAKAGE VISUAL ANTAR SPLIT (dataset baru)")
print(f"{'='*60}")

split_names = ["train", "val", "test"]
cls_folders = [c for c in os.listdir(os.path.join(OUTPUT_ROOT, "train"))]

for cls in cls_folders:
    print(f"\n--- Kelas: {cls} ---")
    split_hashes = {}
    for split in split_names:
        folder = os.path.join(OUTPUT_ROOT, split, cls)
        split_hashes[split] = compute_hashes(folder)
        print(f"  {split}: {len(split_hashes[split])} gambar")

    for i in range(len(split_names)):
        for j in range(i + 1, len(split_names)):
            s1, s2 = split_names[i], split_names[j]
            h1, h2 = split_hashes[s1], split_hashes[s2]
            if not h1 or not h2:
                continue
            found = []
            for f1, hash1 in h1.items():
                for f2, hash2 in h2.items():
                    dist = hash1 - hash2
                    if dist <= HASH_THRESHOLD:
                        found.append((f1, f2, dist))
            print(f"  [{s1} <-> {s2}] gambar mirip ditemukan: {len(found)}")
            if found:
                for f1, f2, dist in found[:5]:
                    print(f"    {s1}/{f1} <-> {s2}/{f2} (distance={dist})")


CEK LEAKAGE VISUAL ANTAR SPLIT (dataset baru)

--- Kelas: Anemia ---
  train: 788 gambar
  val: 102 gambar
  test: 99 gambar
  [train <-> val] gambar mirip ditemukan: 11
    train/Mata-anemia_original_19-jpg_61b624d5-2f15-4ef5-aeaa-abab00b22c3d_jpg.rf.58aff0d1decb9aeb1854764746b55e06.jpg <-> val/Mata-anemia_original_19-jpg_61f634c0-3d64-484c-90b2-d9fcdbb7bd4d_jpg.rf.50941da08c3235d4e1a0e87f605ec037.jpg (distance=0)
    train/Konjungtiva-Anemia_original_014_palpebral-png_518932f6-225a-4d23-8ea8-3a1f62a1aae6_png.rf.eac2eca338bd610ea4f763a64eb6b5a2.jpg <-> val/Konjungtiva-Anemia_original_014_palpebral-png_03fa32c6-5db6-4d5a-85ab-c97dbcc820ce_png.rf.8e0649d1680ed513d6bb880545755be2.jpg (distance=4)
    train/Konjungtiva-Anemia_original_20200214_122915_palpebral-png_22bec136-bc2f-4aa4-801b-5715778c7513_png.rf.174c677d990382622da7a1f3cd550dba.jpg <-> val/Konjungtiva-Anemia_original_20200214_122915_palpebral-png_3b2681ea-4683-4bbd-81c6-435e7f1ce2da_png.rf.c205053eeb44f47d03e140510c028ac5.jpg

In [ ]:
from PIL import ImageEnhance, ImageOps

train_dir = os.path.join(OUTPUT_ROOT, 'train')
counts = {}
for cls in os.listdir(train_dir):
    cls_dir = os.path.join(train_dir, cls)
    counts[cls] = len([f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])

print("=== Distribusi train SEBELUM oversampling ===")
for cls, n in counts.items():
    print(f"  {cls}: {n}")

max_count = max(counts.values())

def augment_variant(img, seed):
    random.seed(seed)
    if random.random() < 0.5:
        img = ImageOps.mirror(img)
    angle = random.uniform(-15, 15)
    img = img.rotate(angle, expand=True, fillcolor=(255, 255, 255))
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.92, 1.08))
    img = ImageEnhance.Color(img).enhance(random.uniform(0.92, 1.08))
    return img

for cls, n in counts.items():
    deficit = max_count - n
    if deficit <= 0:
        continue
    cls_dir = os.path.join(train_dir, cls)
    original_files = [f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    print(f"\nOversampling '{cls}': +{deficit} gambar sintetis...")
    for i in range(deficit):
        src_name = original_files[i % len(original_files)]
        try:
            img = Image.open(os.path.join(cls_dir, src_name)).convert("RGB")
            aug = augment_variant(img, seed=i)
            aug.save(os.path.join(cls_dir, f"aug_{i}_{src_name}"))
        except Exception as e:
            print(f"  Gagal augment {src_name}: {e}")

print("\n=== Distribusi train SETELAH oversampling ===")
for cls in os.listdir(train_dir):
    cls_dir = os.path.join(train_dir, cls)
    n = len([f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    print(f"  {cls}: {n}")

=== Distribusi train SEBELUM oversampling ===
  Anemia: 788
  Normal: 792

Oversampling 'Anemia': +4 gambar sintetis...

=== Distribusi train SETELAH oversampling ===
  Anemia: 792
  Normal: 792


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')

results = model.train(
    data=OUTPUT_ROOT,
    epochs=50,
    imgsz=224,
    patience=15,
    degrees=15,
    hsv_h=0.01,
    hsv_s=0.15,
    hsv_v=0.2,
    fliplr=0.5,
    erasing=0.3,
    project='Palepal_Project',
    name='klasifikasi_konjungtiva_v3_newdata'
)

print("Urutan kelas model konjungtiva:", model.names)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_konjungtiva_od_cls, degrees=15, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.3, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hs

In [ ]:
metrics = model.val(data=OUTPUT_ROOT, split="test")
print("\nTop-1 accuracy di TEST set:", metrics.top1)

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/dataset_konjungtiva_od_cls/train... found 1584 images in 2 classes ✅ 
val: /content/dataset_konjungtiva_od_cls/val... found 196 images in 2 classes ✅ 
test: /content/dataset_konjungtiva_od_cls/test... found 197 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.2±0.3 ms, read: 108.0±33.2 MB/s, size: 2.7 KB)
test: Scanning /content/dataset_konjungtiva_od_cls/test... 197 images, 0 corrupt: 100% ━━━━━━━━━━━━ 197/197 5.3Kit/s 0.0s
test: New cache created: /content/dataset_konjungtiva_od_cls/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 13/13 9.9it/s 1.3s
                   all       0.98          1
Speed: 0.1ms preprocess, 3.8ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val

Top-1 accuracy di TEST set: 0.9796954393386841


In [ ]:
best_path = str(results.save_dir / "weights" / "best.pt")
get_ipython().system(f'yolo task=classify mode=export model={best_path} format=tflite imgsz=224')

latest_train_dir = str(results.save_dir)
print(f"\nFolder hasil training: {latest_train_dir}")

shutil.make_archive('Model_Palepal_Konjungtiva_CLS_v3', 'zip', latest_train_dir)
print("Selesai! File 'Model_Palepal_Konjungtiva_CLS_v3.zip' siap didownload.")

print("\nSELESAI. Cek folder tersebut untuk:")
print("- results.png, confusion_matrix.png")
print("- weights/best.pt dan hasil export .tflite")

WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert/
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs

PyTorch: starting from '/content/runs/classify/Palepal_Project/klasifikasi_konjungtiva_v3_newdata/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 2) (2.8 MB)
requirements: Ultralytics requirements ['litert-torch>=0.9.0', 'ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 85 packages in 661ms
Prepared 15 packages in 4.91s
Uninstalled 3 packages in 43ms
Installed 15 packages in 62ms
 + ai-

In [ ]:
import os
import imagehash
from PIL import Image

OUTPUT_ROOT = "/content/dataset_konjungtiva_od_cls"
HASH_THRESHOLD = 5

def compute_hashes(folder):
    hashes = {}
    if not os.path.exists(folder):
        return hashes
    for f in os.listdir(folder):
        if not f.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        try:
            hashes[f] = imagehash.phash(Image.open(os.path.join(folder, f)))
        except Exception as e:
            print(f"Gagal baca {f}: {e}")
    return hashes

print(f"{'='*60}")
print("CEK LEAKAGE VISUAL ANTAR SPLIT (KONJUNGTIVA - DATASET BARU)")
print(f"{'='*60}")

split_names = ["train", "val", "test"]
class_names = os.listdir(os.path.join(OUTPUT_ROOT, "train"))

total_leak = 0
for cls in class_names:
    print(f"\n--- Kelas: {cls} ---")
    split_hashes = {}
    for split in split_names:
        folder = os.path.join(OUTPUT_ROOT, split, cls)
        split_hashes[split] = compute_hashes(folder)
        print(f"  {split}: {len(split_hashes[split])} gambar")

    for i in range(len(split_names)):
        for j in range(i + 1, len(split_names)):
            s1, s2 = split_names[i], split_names[j]
            h1, h2 = split_hashes[s1], split_hashes[s2]
            if not h1 or not h2:
                continue
            found = []
            for f1, hash1 in h1.items():
                for f2, hash2 in h2.items():
                    dist = hash1 - hash2
                    if dist <= HASH_THRESHOLD:
                        found.append((f1, f2, dist))
            total_leak += len(found)
            print(f"  [{s1} <-> {s2}] gambar mirip ditemukan: {len(found)}")
            if found:
                print(f"    Contoh (max 5):")
                for f1, f2, dist in found[:5]:
                    print(f"      {s1}/{f1}  <->  {s2}/{f2}   (distance={dist})")

print(f"\n\nTOTAL pasangan mirip ditemukan di seluruh split: {total_leak}")
if total_leak > 0:
    print("PERINGATAN: Ada indikasi leakage. Akurasi 98% di atas kemungkinan besar tidak jujur.")
else:
    print("Tidak ada leakage visual terdeteksi. Akurasi 98% lebih bisa dipercaya.")

CEK LEAKAGE VISUAL ANTAR SPLIT (KONJUNGTIVA - DATASET BARU)

--- Kelas: Anemia ---
  train: 792 gambar
  val: 102 gambar
  test: 99 gambar
  [train <-> val] gambar mirip ditemukan: 11
    Contoh (max 5):
      train/Mata-anemia_original_19-jpg_61b624d5-2f15-4ef5-aeaa-abab00b22c3d_jpg.rf.58aff0d1decb9aeb1854764746b55e06.jpg  <->  val/Mata-anemia_original_19-jpg_61f634c0-3d64-484c-90b2-d9fcdbb7bd4d_jpg.rf.50941da08c3235d4e1a0e87f605ec037.jpg   (distance=0)
      train/Konjungtiva-Anemia_original_014_palpebral-png_518932f6-225a-4d23-8ea8-3a1f62a1aae6_png.rf.eac2eca338bd610ea4f763a64eb6b5a2.jpg  <->  val/Konjungtiva-Anemia_original_014_palpebral-png_03fa32c6-5db6-4d5a-85ab-c97dbcc820ce_png.rf.8e0649d1680ed513d6bb880545755be2.jpg   (distance=4)
      train/Konjungtiva-Anemia_original_20200214_122915_palpebral-png_22bec136-bc2f-4aa4-801b-5715778c7513_png.rf.174c677d990382622da7a1f3cd550dba.jpg  <->  val/Konjungtiva-Anemia_original_20200214_122915_palpebral-png_3b2681ea-4683-4bbd-81c6-435e7f1

In [ ]:
import os
import re
import shutil
import random
from PIL import Image, ImageEnhance, ImageOps

OLD_ROOT = "/content/dataset_konjungtiva_od_cls"          # hasil konversi yang sudah ada (leaked)
NEW_ROOT = "/content/dataset_konjungtiva_od_cls_fixed"    # hasil setelah regroup+resplit

for p in [NEW_ROOT]:
    if os.path.exists(p):
        shutil.rmtree(p)

UUID_PATTERN = re.compile(
    r'_[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}_.*$'
)

def extract_source_id(filename):
    """
    Buang bagian UUID augmentasi Roboflow + hash, sisakan identitas
    foto sumber asli. Kalau pola UUID tidak ketemu, fallback pakai
    nama file utuh (dianggap grup sendiri).
    """
    stripped = UUID_PATTERN.sub('', filename)
    return stripped if stripped else filename

In [ ]:
old_splits = ['train', 'val', 'test']
source_data = {}   # (kelas, source_id) -> [fullpath, ...]
unmatched = 0

for split in old_splits:
    split_dir = os.path.join(OLD_ROOT, split)
    if not os.path.exists(split_dir):
        continue
    for cls in os.listdir(split_dir):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            # PENTING: skip file hasil oversampling dari run sebelumnya,
            # supaya tidak ikut di-regroup dan di-oversample lagi (double).
            if fname.startswith("aug_"):
                continue
            fpath = os.path.join(cls_dir, fname)
            source_id = extract_source_id(fname)
            if source_id == fname:
                unmatched += 1
            key = (cls, source_id)
            source_data.setdefault(key, []).append(fpath)

print(f"Total grup foto sumber unik: {len(source_data)}")
print(f"File yang tidak match pola UUID (fallback grup sendiri): {unmatched}")

n_dupe_groups = sum(1 for files in source_data.values() if len(files) > 1)
print(f"Grup berisi lebih dari 1 augmentasi (dari foto sumber sama): {n_dupe_groups}")

Total grup foto sumber unik: 372
File yang tidak match pola UUID (fallback grup sendiri): 0
Grup berisi lebih dari 1 augmentasi (dari foto sumber sama): 351


In [ ]:
random.seed(42)

groups_by_class = {}
for (cls, source_id), files in source_data.items():
    groups_by_class.setdefault(cls, []).append(files)

TRAIN_RATIO, VAL_RATIO = 0.8, 0.1

for cls, groups in groups_by_class.items():
    random.shuffle(groups)
    n = len(groups)
    n_train = max(1, int(n * TRAIN_RATIO))
    n_val = max(1, int(n * VAL_RATIO))

    train_groups = groups[:n_train]
    val_groups = groups[n_train:n_train + n_val]
    test_groups = groups[n_train + n_val:]

    print(f"\nKelas '{cls}': total_grup={n}, train={len(train_groups)}, "
          f"val={len(val_groups)}, test={len(test_groups)}")

    for split_name, split_groups in [('train', train_groups), ('val', val_groups), ('test', test_groups)]:
        out_dir = os.path.join(NEW_ROOT, split_name, cls)
        os.makedirs(out_dir, exist_ok=True)
        for files in split_groups:
            for fpath in files:
                shutil.copy(fpath, os.path.join(out_dir, os.path.basename(fpath)))

print("\n=== Distribusi akhir dataset baru (per file) ===")
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(NEW_ROOT, split)
    if not os.path.exists(split_dir):
        continue
    print(f"\n[{split}]")
    for cls in os.listdir(split_dir):
        n = len(os.listdir(os.path.join(split_dir, cls)))
        print(f"  {cls}: {n} gambar")


Kelas 'Anemia': total_grup=159, train=127, val=15, test=17

Kelas 'Normal': total_grup=213, train=170, val=21, test=22

=== Distribusi akhir dataset baru (per file) ===

[train]
  Anemia: 797 gambar
  Normal: 796 gambar

[val]
  Anemia: 89 gambar
  Normal: 94 gambar

[test]
  Anemia: 103 gambar
  Normal: 94 gambar


In [ ]:
print("\n=== Verifikasi ulang: cek source-id leakage di dataset baru ===")
check = {}
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(NEW_ROOT, split)
    for cls in os.listdir(split_dir):
        for fname in os.listdir(os.path.join(split_dir, cls)):
            sid = extract_source_id(fname)
            check.setdefault((cls, sid), set()).add(split)

leak_found = False
for (cls, sid), splits_found in check.items():
    if len(splits_found) > 1:
        leak_found = True
        print(f"  masih bocor: [{cls}] {sid} -> {sorted(splits_found)}")

print("Aman, tidak ada leakage." if not leak_found else "MASIH ADA LEAKAGE, cek regex-nya.")


=== Verifikasi ulang: cek source-id leakage di dataset baru ===
Aman, tidak ada leakage.


In [ ]:
train_dir = os.path.join(NEW_ROOT, 'train')
counts = {cls: len(os.listdir(os.path.join(train_dir, cls))) for cls in os.listdir(train_dir)}
print("\n=== Distribusi train SEBELUM oversampling ===", counts)

max_count = max(counts.values())

def augment_variant(img, seed):
    random.seed(seed)
    if random.random() < 0.5:
        img = ImageOps.mirror(img)
    img = img.rotate(random.uniform(-15, 15), expand=True, fillcolor=(255, 255, 255))
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.92, 1.08))
    img = ImageEnhance.Color(img).enhance(random.uniform(0.92, 1.08))
    return img

for cls, n in counts.items():
    deficit = max_count - n
    if deficit <= 0:
        continue
    cls_dir = os.path.join(train_dir, cls)
    original_files = os.listdir(cls_dir)
    for i in range(deficit):
        src_name = original_files[i % len(original_files)]
        try:
            img = Image.open(os.path.join(cls_dir, src_name)).convert("RGB")
            aug = augment_variant(img, seed=i)
            aug.save(os.path.join(cls_dir, f"aug_{i}_{src_name}"))
        except Exception as e:
            print(f"Gagal augment {src_name}: {e}")

print("=== Distribusi train SETELAH oversampling ===")
for cls in os.listdir(train_dir):
    print(f"  {cls}: {len(os.listdir(os.path.join(train_dir, cls)))}")


=== Distribusi train SEBELUM oversampling === {'Anemia': 797, 'Normal': 796}
=== Distribusi train SETELAH oversampling ===
  Anemia: 797
  Normal: 797


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')
results = model.train(
    data=NEW_ROOT,
    epochs=50,
    imgsz=224,
    patience=15,
    hsv_h=0.01, hsv_s=0.15, hsv_v=0.2,
    fliplr=0.5, degrees=15, erasing=0.3,
    project='Palepal_Project',
    name='klasifikasi_konjungtiva_v4_fixed'
)

print("Urutan kelas:", model.names)

metrics = model.val(data=NEW_ROOT, split="test")
print("\nTop-1 accuracy TEST (setelah fix leakage):", metrics.top1)

best_path = str(results.save_dir / "weights" / "best.pt")
get_ipython().system(f'yolo task=classify mode=export model={best_path} format=tflite imgsz=224')

shutil.make_archive('Model_Palepal_Konjungtiva_CLS_v4_fixed', 'zip', str(results.save_dir))
print("Selesai! File 'Model_Palepal_Konjungtiva_CLS_v4_fixed.zip' siap didownload.")

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_konjungtiva_od_cls_fixed, degrees=15, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.3, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.15, hsv_v=0.2, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=klasifikasi_konjungtiva_v4_fixed, nbs=64, nms=False, opset=None, optim